In [1]:
from together import Together
from rdflib import Graph, RDF, RDFS, OWL, BNode
import csv, random, re, time, os
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================
EVAL_CSV      = "CQ_EVALUATION_ONLY.csv"
CCO_TTL       = "CCO/CCO (V1).ttl"
SHAPES_FILE   = "CCO/cco_shapes.ttl"
OUTPUT_DIR    = Path("abox_output")
OUTPUT_DIR.mkdir(exist_ok=True)

SAMPLE_PCT    = 0.10
RANDOM_SEED   = 42
MAX_RETRIES   = 3
REQUEST_DELAY = 0.3
MODEL         = "deepseek-ai/DeepSeek-V3.1"
TEST_MODE     = False    # True = first 10 only | False = full run
TEST_LIMIT    = 10

TOGETHER_API_KEY = ""  # paste your key here

CCO_NAMESPACE  = "https://www.w3id.org/cco/cco#"
ABOX_NAMESPACE = "https://www.w3id.org/cco/abox#"

# ============================================================
# AUTH
# ============================================================
api_key = (os.environ.get("TOGETHER_API_KEY") or TOGETHER_API_KEY).strip()
if not api_key:
    raise ValueError("Missing Together API key")

client = Together(api_key=api_key)

# ============================================================
# STEP 1 — Load CCO Schema from TTL
# ============================================================
def _local_name(uri):
    if "#" in uri:
        return uri.split("#")[-1]
    return uri.rstrip("/").split("/")[-1]

def get_restriction_label(g, node):
    on_property = g.value(node, OWL.onProperty)
    prop_label  = g.value(on_property, RDFS.label) if on_property else None
    prop_name   = (
        str(prop_label) if prop_label
        else (_local_name(str(on_property)) if on_property else "?")
    )
    some_values = g.value(node, OWL.someValuesFrom)
    on_class    = g.value(node, OWL.onClass)
    on_range    = g.value(node, OWL.onDataRange)
    min_card    = g.value(node, OWL.minQualifiedCardinality)
    max_card    = g.value(node, OWL.maxQualifiedCardinality)
    exact_card  = g.value(node, OWL.qualifiedCardinality)

    target = None
    if on_class:
        if isinstance(on_class, BNode):
            target = "[complex]"
        else:
            tl = g.value(on_class, RDFS.label)
            target = str(tl) if tl else _local_name(str(on_class))
    elif on_range:
        target = _local_name(str(on_range))

    if some_values:
        if isinstance(some_values, BNode):
            return f"{prop_name} some [complex]"
        sv = str(g.value(some_values, RDFS.label) or _local_name(str(some_values)))
        return f"{prop_name} some {sv}"
    if exact_card: return f"{prop_name} exactly {exact_card} {target or ''}".strip()
    if min_card:   return f"{prop_name} min {min_card} {target or ''}".strip()
    if max_card:   return f"{prop_name} max {max_card} {target or ''}".strip()
    return f"restriction on {prop_name}"

def load_cco_schema(ttl_path):
    g = Graph()
    g.parse(ttl_path, format="turtle")

    classes, constraints = [], {}
    obj_props, dat_props = [], []

    for cls in g.subjects(RDF.type, OWL.Class):
        if not str(cls).startswith("http"):
            continue
        label    = g.value(cls, RDFS.label)
        cls_name = str(label) if label else _local_name(str(cls))
        cls_constraints, super_names = [], []

        for sc in g.objects(cls, RDFS.subClassOf):
            if isinstance(sc, BNode):
                ix = g.value(sc, OWL.intersectionOf)
                if ix:
                    for item in list(g.items(ix)):
                        if isinstance(item, BNode):
                            cls_constraints.append(
                                f"AND {get_restriction_label(g, item)}"
                            )
                else:
                    cls_constraints.append(get_restriction_label(g, sc))
            else:
                sl = g.value(sc, RDFS.label)
                super_names.append(str(sl) if sl else _local_name(str(sc)))

        classes.append(cls_name)
        constraints[cls_name] = {
            "subclasses":  super_names,
            "constraints": cls_constraints,
        }

    classes.sort()

    for prop in g.subjects(RDF.type, OWL.ObjectProperty):
        label  = g.value(prop, RDFS.label)
        domain = g.value(prop, RDFS.domain)
        range_ = g.value(prop, RDFS.range)
        obj_props.append({
            "name":   str(label) if label else _local_name(str(prop)),
            "domain": str(g.value(domain, RDFS.label) or _local_name(str(domain))) if domain else "None",
            "range":  str(g.value(range_, RDFS.label) or _local_name(str(range_))) if range_ else "None",
        })

    for prop in g.subjects(RDF.type, OWL.DatatypeProperty):
        label  = g.value(prop, RDFS.label)
        domain = g.value(prop, RDFS.domain)
        range_ = g.value(prop, RDFS.range)
        dat_props.append({
            "name":   str(label) if label else _local_name(str(prop)),
            "domain": str(g.value(domain, RDFS.label) or _local_name(str(domain))) if domain else "None",
            "range":  _local_name(str(range_)) if range_ else "None",
        })

    schema_str = "CCO CLASSES (with constraints):\n"
    for cls_name in classes:
        info = constraints.get(cls_name, {})
        schema_str += f"- {cls_name}"
        if info.get("subclasses"):
            schema_str += f" (subClassOf: {', '.join(info['subclasses'])})"
        schema_str += "\n"
        for c in info.get("constraints", []):
            schema_str += f"    Constraint: {c}\n"

    schema_str += "\nOBJECT PROPERTIES (Domain -> Range):\n"
    for p in obj_props:
        schema_str += f"- {p['name']}: {p['domain']} -> {p['range']}\n"

    schema_str += "\nDATATYPE PROPERTIES (Domain -> Range):\n"
    for p in dat_props:
        schema_str += f"- {p['name']}: {p['domain']} -> {p['range']}\n"

    print(f"CCO loaded: {len(classes)} classes, "
          f"{len(obj_props)} object properties, "
          f"{len(dat_props)} datatype properties")
    return schema_str

print("Loading CCO schema...")
SCHEMA_STR = load_cco_schema(CCO_TTL)

# ============================================================
# STEP 2 — Prompt Builders
# ============================================================
SYSTEM_MSG = (
    "You are a compliance ontology engineer. "
    "Respond only in English. "
    "Return only valid content — no explanation, no markdown."
)

def build_abox_prompt(row):
    return f"""Generate a minimal valid RDF/Turtle ABox using ONLY the CCO schema below.
The ABox must represent the normative structure so the competency question can be answered.

CCO Schema:
{SCHEMA_STR}

STRICT RULES:
1. Always include these prefixes:
   @prefix cco:  <{CCO_NAMESPACE}> .
   @prefix abox: <{ABOX_NAMESPACE}> .
   @prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .
   @prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

2. Every cco:Regulation MUST have cco:hasValidityStart (xsd:date)
3. Every cco:Regulation MUST have at least one cco:specifiesNorm
4. Every cco:RoleHolding MUST have cco:hasRole and cco:hasStartTime
5. Every cco:Exception MUST have cco:modifiesNorm and cco:hasCondition
6. Use abox: prefix for ALL instances
7. Keep minimal — only what is needed to answer the CQ
8. Return ONLY valid Turtle — no explanation, no markdown fences
9. Every Norm MUST be linked from a cco:Regulation via cco:specifiesNorm
10. Always declare deontic instances with BOTH their type AND cco:Norm:
    abox:MyObligation  a cco:Obligation,  cco:Norm .
    abox:MyPermission  a cco:Permission,  cco:Norm .
    abox:MyProhibition a cco:Prohibition, cco:Norm .

Competency Question : {row.get('question', '')}
Clause             : {row.get('clause', '')}
Regulatory Text    : {row.get('excerpt', '')}
CCO Elements       : {row.get('cco_elements', '')}
Domain             : {row.get('domain', '')}

Generate Turtle ABox:"""


def build_sparql_prompt(row, abox_ttl):
    return f"""You are a compliance ontology engineer. Respond only in English.

Given the ABox below and the competency question, generate a SPARQL SELECT query
that answers the question by querying the ABox.

Prefixes:
PREFIX cco:  <{CCO_NAMESPACE}>
PREFIX abox: <{ABOX_NAMESPACE}>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

ABox (Turtle):
{abox_ttl}

Competency Question: {row.get('question', '')}
Domain: {row.get('domain', '')}

Rules:
1. Use SELECT query only
2. Use the exact IRIs from the ABox above
3. Return ONLY the SPARQL query — no explanation, no markdown fences

Generate SPARQL query:"""

# ============================================================
# STEP 3 — Sampling
# ============================================================
def load_and_sample(csv_path, pct=0.10, seed=42, test_mode=True, test_limit=10):
    random.seed(seed)
    with open(csv_path, "r", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    yes_rows = [r for r in rows if r.get("llm_assessment") == "Yes"]

    by_domain = {}
    for r in yes_rows:
        by_domain.setdefault(r.get("domain", ""), []).append(r)

    sample = []
    print("Sampling:")
    for domain, domain_rows in sorted(by_domain.items()):
        n = max(1, round(len(domain_rows) * pct))
        sampled = random.sample(domain_rows, n)
        sample.extend(sampled)
        print(f"  {domain}: {len(domain_rows)} Yes CQs -> {n} sampled")

    print(f"\n  Total full sample: {len(sample)} CQs")

    if test_mode:
        sample = sample[:test_limit]
        print(f"  TEST MODE: using first {test_limit} CQs only\n")
    else:
        print(f"  FULL RUN: {len(sample)} CQs\n")

    return sample

# ============================================================
# STEP 4 — LLM Call
# ============================================================
def call_llm(prompt, max_tokens=1500, max_retries=MAX_RETRIES):
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_MSG},
                    {"role": "user",   "content": prompt},
                ],
                max_tokens=max_tokens,
                temperature=0.0,
            )
            txt = resp.choices[0].message.content or ""
            txt = re.sub(r"```(?:turtle|ttl|rdf|sparql)?\s*", "", txt)
            txt = re.sub(r"```\s*", "", txt)
            time.sleep(REQUEST_DELAY)
            return txt.strip()
        except Exception as e:
            err = str(e).lower()
            if "429" in err or "rate limit" in err:
                wait = 60 * attempt
                print(f"    Rate limit. Waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"    Attempt {attempt}/{max_retries}: {e}")
                if attempt < max_retries:
                    time.sleep(5 * attempt)
    return ""

# ============================================================
# STEP 5 — SHACL Validation
# ============================================================
def validate_abox(ttl_content, shapes_file):
    try:
        from pyshacl import validate
        import tempfile

        prefixes = (
            f"@prefix cco:  <{CCO_NAMESPACE}> .\n"
            f"@prefix abox: <{ABOX_NAMESPACE}> .\n"
            "@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .\n"
            "@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .\n"
            "@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .\n\n"
        )

        full_ttl = (
            prefixes + ttl_content
            if "@prefix cco:" not in ttl_content[:300]
            else ttl_content
        )

        with tempfile.NamedTemporaryFile(
            mode="w", suffix=".ttl", delete=False, encoding="utf-8"
        ) as f:
            f.write(full_ttl)
            tmp = f.name

        conforms, _, results_text = validate(
            tmp,
            shacl_graph=shapes_file,
            inference="both",
            abort_on_first=False,
        )
        os.unlink(tmp)

        violations = results_text.count("Violation")
        passed     = violations == 0

        return {"conforms": passed, "violations": violations}

    except Exception as e:
        return {"conforms": None, "violations": -1, "details": str(e)[:200]}

# ============================================================
# STEP 6 — SPARQL Execution against ABox
# ============================================================
def execute_sparql(sparql_query, abox_ttl):
    """
    Execute SPARQL SELECT against ABox.
    Returns: "Yes" (results found), "No" (valid but empty), "SPARQL_ERROR"
    """
    try:
        from rdflib import Graph
        import tempfile

        prefixes = (
            f"@prefix cco:  <{CCO_NAMESPACE}> .\n"
            f"@prefix abox: <{ABOX_NAMESPACE}> .\n"
            "@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .\n"
            "@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .\n\n"
        )

        full_ttl = (
            prefixes + abox_ttl
            if "@prefix cco:" not in abox_ttl[:300]
            else abox_ttl
        )

        g = Graph()
        g.parse(data=full_ttl, format="turtle")

        # Add PREFIX declarations if missing
        if "PREFIX cco:" not in sparql_query and "prefix cco:" not in sparql_query.lower():
            sparql_query = (
                f"PREFIX cco:  <{CCO_NAMESPACE}>\n"
                f"PREFIX abox: <{ABOX_NAMESPACE}>\n"
                "PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>\n"
                "PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\n\n"
                + sparql_query
            )

        results = list(g.query(sparql_query))

        if results:
            return "Yes", len(results)
        else:
            return "No", 0

    except Exception as e:
        return "SPARQL_ERROR", str(e)[:150]

# ============================================================
# STEP 7 — Main Pipeline
# ============================================================
def run_pipeline():
    print("=" * 55)
    print("CCO ABox + SPARQL + SHACL Validation Pipeline")
    print(f"Mode: {'TEST MODE' if TEST_MODE else 'FULL RUN'}")
    print("=" * 55)

    sample  = load_and_sample(
        EVAL_CSV, SAMPLE_PCT, RANDOM_SEED,
        test_mode=TEST_MODE, test_limit=TEST_LIMIT
    )
    total   = len(sample)
    results = []

    print(f"Processing {total} CQs...\n")

    for idx, row in enumerate(sample, 1):
        cq_id  = row.get("cq_id", "?")
        domain = row.get("domain", "?")
        pct    = idx / total * 100

        # ── A: Generate ABox ──────────────────────────────
        abox_ttl = call_llm(build_abox_prompt(row), max_tokens=1500)

        if not abox_ttl:
            print(f"  [{idx}/{total}, {pct:.0f}%] {cq_id} ({domain}): EMPTY ABOX")
            results.append({
                "cq_id": cq_id, "domain": domain,
                "question": row.get("question", "")[:80],
                "shacl_status": "ERROR", "violations": -1,
                "sparql_result": "N/A", "sparql_rows": 0,
                "abox_generated": False, "sparql_generated": False,
                "details": "LLM returned empty ABox",
            })
            continue

        # Save ABox TTL
        ttl_path = OUTPUT_DIR / f"{cq_id}_{domain.replace(' ', '_')}.ttl"
        ttl_path.write_text(abox_ttl, encoding="utf-8")

        # ── B: SHACL Validation ───────────────────────────
        val          = validate_abox(abox_ttl, SHAPES_FILE)
        shacl_status = (
            "PASS"  if val["conforms"] is True  else
            "ERROR" if val["conforms"] is None  else
            "FAIL"
        )

        # ── C: Generate SPARQL (always) ───────────────────
        sparql_query = call_llm(
            build_sparql_prompt(row, abox_ttl), max_tokens=600
        )

        # Save SPARQL
        sparql_result = "N/A"
        sparql_rows   = 0
        sparql_detail = ""

        if sparql_query:
            sparql_path = OUTPUT_DIR / f"{cq_id}_{domain.replace(' ', '_')}.sparql"
            sparql_path.write_text(sparql_query, encoding="utf-8")

            # ── D: Execute SPARQL against ABox ────────────
            sparql_result, sparql_rows = execute_sparql(sparql_query, abox_ttl)

            if sparql_result == "SPARQL_ERROR":
                sparql_detail = str(sparql_rows)[:100]
                sparql_rows   = 0

        print(
            f"  [{idx}/{total}, {pct:.0f}%] {cq_id} ({domain}): "
            f"SHACL={shacl_status} | SPARQL={sparql_result} "
            f"(rows={sparql_rows}) | V={val['violations']}"
        )

        results.append({
            "cq_id":            cq_id,
            "domain":           domain,
            "question":         row.get("question", "")[:80],
            "shacl_status":     shacl_status,
            "violations":       val["violations"],
            "sparql_result":    sparql_result,
            "sparql_rows":      sparql_rows,
            "abox_generated":   True,
            "sparql_generated": bool(sparql_query),
            "details":          sparql_detail or val.get("details", ""),
        })

    # ── Save results CSV ──────────────────────────────────
    results_path = OUTPUT_DIR / "validation_results.csv"
    with results_path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(
            f,
            fieldnames=[
                "cq_id", "domain", "question",
                "shacl_status", "violations",
                "sparql_result", "sparql_rows",
                "abox_generated", "sparql_generated", "details"
            ],
            extrasaction="ignore",
        )
        w.writeheader()
        for r in results:
            w.writerow(r)

    # ── Summary ───────────────────────────────────────────
    generated      = sum(1 for r in results if r["abox_generated"])
    shacl_pass     = sum(1 for r in results if r["shacl_status"] == "PASS")
    shacl_fail     = sum(1 for r in results if r["shacl_status"] == "FAIL")
    sparql_yes     = sum(1 for r in results if r["sparql_result"] == "Yes")
    sparql_no      = sum(1 for r in results if r["sparql_result"] == "No")
    sparql_err     = sum(1 for r in results if r["sparql_result"] == "SPARQL_ERROR")

    by_domain = {}
    for r in results:
        d = r["domain"]
        by_domain.setdefault(d, {
            "shacl_pass": 0, "shacl_fail": 0,
            "sparql_yes": 0, "sparql_no": 0,
            "sparql_err": 0, "total": 0
        })
        by_domain[d]["total"] += 1
        if r["shacl_status"] == "PASS":     by_domain[d]["shacl_pass"] += 1
        elif r["shacl_status"] == "FAIL":   by_domain[d]["shacl_fail"] += 1
        if r["sparql_result"] == "Yes":     by_domain[d]["sparql_yes"] += 1
        elif r["sparql_result"] == "No":    by_domain[d]["sparql_no"]  += 1
        elif r["sparql_result"] == "SPARQL_ERROR": by_domain[d]["sparql_err"] += 1

    print("\n" + "=" * 55)
    print("SUMMARY")
    print("=" * 55)
    print(f"Total sampled    : {total}")
    print(f"ABox generated   : {generated}")
    print(f"SHACL PASS       : {shacl_pass}")
    print(f"SHACL FAIL       : {shacl_fail}")
    print(f"SPARQL Yes       : {sparql_yes}")
    print(f"SPARQL No        : {sparql_no}")
    print(f"SPARQL Error     : {sparql_err}")
    print(f"\nBy domain:")
    for d, c in sorted(by_domain.items()):
        sp = c["shacl_pass"] / c["total"] * 100 if c["total"] else 0
        print(
            f"  {d:25s}: "
            f"SHACL {c['shacl_pass']}/{c['total']} PASS ({sp:.0f}%) | "
            f"SPARQL Yes={c['sparql_yes']} No={c['sparql_no']} Err={c['sparql_err']}"
        )
    print(f"\nResults : {results_path}")
    print(f"TTL dir : {OUTPUT_DIR}/")
    return results

# ============================================================
# RUN
# ============================================================
results = run_pipeline()

Loading CCO schema...
CCO loaded: 16 classes, 14 object properties, 7 datatype properties
CCO ABox + SPARQL + SHACL Validation Pipeline
Mode: FULL RUN
Sampling:
  Data Protection: 589 Yes CQs -> 59 sampled
  Education: 481 Yes CQs -> 48 sampled
  Finance: 250 Yes CQs -> 25 sampled
  Healthcare: 236 Yes CQs -> 24 sampled

  Total full sample: 156 CQs
  FULL RUN: 156 CQs

Processing 156 CQs...

  [1/156, 1%] CQ122 (Data Protection): SHACL=PASS | SPARQL=Yes (rows=1) | V=0
  [2/156, 1%] CQ031 (Data Protection): SHACL=PASS | SPARQL=Yes (rows=1) | V=0
  [3/156, 2%] CQ297 (Data Protection): SHACL=PASS | SPARQL=No (rows=0) | V=0
  [4/156, 3%] CQ263 (Data Protection): SHACL=PASS | SPARQL=Yes (rows=1) | V=0
  [5/156, 3%] CQ239 (Data Protection): SHACL=PASS | SPARQL=Yes (rows=1) | V=0
  [6/156, 4%] CQ150 (Data Protection): SHACL=PASS | SPARQL=Yes (rows=1) | V=0
  [7/156, 4%] CQ112 (Data Protection): SHACL=PASS | SPARQL=Yes (rows=1) | V=0
  [8/156, 5%] CQ574 (Data Protection): SHACL=PASS | SPARQL=